<a href="https://colab.research.google.com/github/brendonhuynhbp-hub/gt-markets/blob/main/notebooks/artefacts_prepare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
# ==== Normalize 'extracted' (daily/weekly) into app-ready artefacts ====
# - Reads predictions & summaries you've already produced
# - Derives signals from prob_up (0.5) if needed
# - Computes baseline metrics (Return_Ann, Vol_Ann, Sharpe, MaxDD)
# - Copies/creates figs
# - Writes clean structure the app expects

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import re, glob, shutil, math

# -------- CONFIG (edit paths if different) --------
DRIVE_ROOT  = Path("/content/drive/MyDrive/gt-markets")
SRC_ROOT    = DRIVE_ROOT / "app-demo" / "extracted"              # you said data lives here
DST_ROOT    = DRIVE_ROOT / "AppDemo" / "artefacts_from_mainnb"   # clean output for the app
DATA_DIR    = DRIVE_ROOT / "data" / "processed"                  # fallback for prices if needed

ASSETS      = ["GOLD", "BTC", "OIL", "USDCNY"]                   # normalize to these labels
FREQ_DIRS   = {"daily":"D", "weekly":"W"}
ANNUALIZATION = {"D":252, "W":52}
COST_BPS    = 5   # simple trading cost assumption

# Map from label -> price column in processed data (fallback if preds don't have Close)
PRICE_COL = {
    "GOLD":   "GC=F Close",
    "BTC":    "BTC-USD Close",
    "OIL":    "CL=F Close",
    "USDCNY": "USDCNY=X Close",
}

# ---- Helpers ----
def ensure_clean_dir(p: Path):
    if p.exists(): shutil.rmtree(p)
    p.mkdir(parents=True, exist_ok=True)

def latest_processed_csv() -> Path:
    files = sorted(DATA_DIR.glob("merged_financial_trends_data_*.csv"))
    if not files:
        raise FileNotFoundError("No processed merged_financial_trends_data_*.csv in data/processed")
    return files[-1]

def infer_asset_from_name(name: str) -> str|None:
    n = name.lower()
    for a in ASSETS:
        if a.lower() in n:
            return a
    # common tickers as backup
    tick2asset = {"gc=f":"GOLD","btc-usd":"BTC","cl=f":"OIL","usdcny=x":"USDCNY"}
    for t,a in tick2asset.items():
        if t in n: return a
    return None

def infer_strategy_from_name(name: str) -> str|None:
    n = name.lower()
    # common model names / strategies
    for s in ["rf","xgb","xgboost","mlp","lstm","gru","lr","svm","ridge","stack","ensemble","calibrated","randforest","randomforest"]:
        if s in n:
            return s.upper() if s in ["rf","xgb","mlp","lstm","gru","lr","svm"] else s
    # generic aliases
    if "trend" in n: return "TREND"
    if "breakout" in n or "donch" in n: return "BREAKOUT"
    if "mean" in n and "rev" in n: return "MEANREV"
    return None

def to_equity(close: pd.Series, pos: pd.Series, freq: str):
    ret = close.pct_change().fillna(0.0)
    pos = pos.fillna(0.0).clip(0,1)
    trades = pos.diff().abs().fillna(0.0)
    cost = trades * (COST_BPS / 1e4)
    strat = (pos.shift(1) * ret) - cost
    eq = (1 + strat).cumprod()
    ann = ANNUALIZATION[freq]
    mu = strat.mean() * ann
    sd = strat.std() * (ann ** 0.5) if strat.std() > 0 else np.nan
    sharpe = mu / sd if sd and sd > 0 else np.nan
    maxdd = (eq / eq.cummax() - 1).min()
    return eq, {"Return_Ann": mu, "Vol_Ann": sd, "Sharpe": sharpe, "MaxDD": maxdd}

def plot_eq(eq: pd.Series, title: str, outpath: Path):
    fig, ax = plt.subplots(figsize=(6,3))
    eq.plot(ax=ax); ax.set_title(title); ax.grid(True, alpha=0.3)
    fig.savefig(outpath, dpi=150, bbox_inches="tight"); plt.close(fig)

# ---- Prepare destination ----
ensure_clean_dir(DST_ROOT)

# Load fallback prices (if needed)
price_df = None
try:
    price_df = pd.read_csv(latest_processed_csv(), parse_dates=["Date"]).set_index("Date").sort_index()
except Exception:
    pass

# ---- Walk source folders (daily/weekly) ----
for freq_dir, F in FREQ_DIRS.items():
    src = SRC_ROOT / freq_dir
    if not src.exists():
        print(f"[WARN] Missing {src}")
        continue

    # Collect candidate CSVs: predictions, metrics, possibly signals already
    csvs = list(src.rglob("*.csv"))
    pngs = list(src.rglob("*.png")) + list(src.rglob("*.jpg"))

    # Prepare per-asset accumulators
    per_asset_metrics = {a: [] for a in ASSETS}
    per_asset_figs = {a: [] for a in ASSETS}

    # 1) If your extracted includes a backtest summary, prefer it for metrics
    summaries = [c for c in csvs if "backtest_summary" in c.name.lower() or "leaderboard" in c.name.lower()]
    summary_df = None
    if summaries:
        try:
            summary_df = pd.read_csv(summaries[0])
        except Exception:
            summary_df = None

    # 2) Handle predictions (derive signals from prob_up threshold 0.5)
    #    Also copy any existing signals_* files directly if present.
    #    Output layout: artefacts/<ASSET>/signals_<STRAT>_<F>.csv
    for c in csvs:
        name = c.name.lower()

        # Already-normalized signals? just map/copy
        if name.startswith("signals_") and name.endswith(".csv"):
            asset = infer_asset_from_name(name) or infer_asset_from_name(str(c))
            if not asset: continue
            out_dir = DST_ROOT / asset; (out_dir/"figs").mkdir(parents=True, exist_ok=True)
            strat = infer_strategy_from_name(name) or "MODEL"
            dst = out_dir / f"signals_{strat}_{F}.csv"
            shutil.copy2(c, dst)
            continue

        # Predictions with prob_up
        if "prob_up" in name or "pred" in name or "prediction" in name:
            asset = infer_asset_from_name(name) or infer_asset_from_name(str(c))
            if not asset: continue
            try:
                dfp = pd.read_csv(c)
            except Exception:
                continue

            # infer date column
            dt_col = None
            for cand in ["Date","date","timestamp","time"]:
                if cand in dfp.columns:
                    dt_col = cand; break

            if dt_col is None:
                # sometimes predictions are saved index-like 0..N with a separate join key -> skip
                continue

            dfp[dt_col] = pd.to_datetime(dfp[dt_col], errors="coerce")
            dfp = dfp.dropna(subset=[dt_col]).sort_values(dt_col)

            # infer prob column
            p_col = None
            for cand in ["prob_up","p_up","prob","proba","prob1","p1"]:
                if cand in dfp.columns:
                    p_col = cand; break
            if p_col is None:
                continue

            # infer Close price column
            close = None
            # 1) in preds file
            for cand in ["Close","close","price","close_adj","adj_close"]:
                if cand in dfp.columns:
                    close = dfp[cand]; break
            # 2) fallback to processed data file
            if close is None and price_df is not None:
                col = PRICE_COL.get(asset)
                if col and col in price_df.columns:
                    close = price_df[col].reindex(dfp[dt_col]).ffill()

            if close is None:
                # cannot compute metrics/equity without price; skip signals creation
                continue

            # derive position & signal
            pos = (dfp[p_col] > 0.5).astype(float)
            sig = pos.diff().fillna(0.0).replace({1.0:1,-1.0:0}).astype(int)

            # write signals
            out_dir = DST_ROOT / asset; (out_dir/"figs").mkdir(parents=True, exist_ok=True)
            strat = infer_strategy_from_name(name) or "MODEL"
            df_sig = pd.DataFrame({
                "Date": dfp[dt_col].values,
                "signal": sig.values,
                "position": pos.astype(int).values,
                "Close": pd.to_numeric(close, errors="coerce").values,
                "strategy": strat
            })
            out_sig = out_dir / f"signals_{strat}_{F}.csv"
            df_sig.to_csv(out_sig, index=False)

            # compute equity + metrics
            close_series = pd.to_numeric(close, errors="coerce")
            close_series.index = pd.to_datetime(dfp[dt_col].values)
            eq, mets = to_equity(close_series, pos, F)
            per_asset_metrics[asset].append({"strategy": strat, **mets})

            # make fig
            fig_path = out_dir / "figs" / f"{strat}_{F}.png"
            plot_eq(eq, f"{asset} — {strat} — {F}", fig_path)
            per_asset_figs[asset].append(fig_path)

    # 3) If a backtest summary exists, also write metrics_baseline_<F>.csv from it (fallback)
    for asset in ASSETS:
        out_dir = DST_ROOT / asset; (out_dir/"figs").mkdir(parents=True, exist_ok=True)
        rows = per_asset_metrics[asset]

        # Append summary_df if available and has usable fields
        if summary_df is not None and "asset" in summary_df.columns:
            sdf = summary_df.copy()
            # normalize asset names to our labels
            sdf["asset_norm"] = sdf["asset"].str.upper().replace({
                "GOLD":"GOLD","BTC":"BTC","OIL":"OIL","USDCNY":"USDCNY"
            })
            sdf = sdf[sdf["asset_norm"] == asset]
            # map a few common metric columns if present
            strat = sdf["model"].astype(str) if "model" in sdf.columns else pd.Series(["MODEL"]*len(sdf))
            eqv  = sdf["final_equity"] if "final_equity" in sdf.columns else None
            # convert equity to simple return if given
            if eqv is not None:
                try:
                    ret_approx = eqv.astype(float) - 1.0
                    for s, r in zip(strat, ret_approx):
                        rows.append({"strategy": str(s), "Return_Ann": float(r)})
                except Exception:
                    pass

        # Write metrics file if any rows
        if rows:
            mdf = pd.DataFrame(rows)
            mdf.insert(0, "asset", asset)
            mdf.insert(1, "freq", F)
            mdf.to_csv(out_dir / f"metrics_baseline_{F}.csv", index=False)
        else:
            # create an empty placeholder to satisfy app diagnostics
            pd.DataFrame(columns=["asset","freq","strategy","Return_Ann","Vol_Ann","Sharpe","MaxDD"])\
              .to_csv(out_dir / f"metrics_baseline_{F}.csv", index=False)

        # If there are loose figs in source, copy a few (optional)
        seen = set(p.name for p in per_asset_figs[asset])
        for p in pngs:
            if infer_asset_from_name(p.name) == asset and p.name not in seen:
                dstp = out_dir / "figs" / p.name
                try: shutil.copy2(p, dstp)
                except: pass

print("✅ Normalized app artefacts written to:", DST_ROOT)
print("Now copy the folder to your repo as AppDemo/artefacts/")


Mounted at /content/drive
✅ Normalized app artefacts written to: /content/drive/MyDrive/gt-markets/AppDemo/artefacts_from_mainnb
Now copy the folder to your repo as AppDemo/artefacts/
